# Week 10 Pair Programming: Pipeline Exploration

## Goal
Build fluency with the Hugging Face `transformers` API by trying different tasks. Each pair tries one task in depth.

> **Note (transformers v5):** Some tasks still have a one-line `pipeline()` shortcut (NER, fill-mask, text-generation). The sequence-to-sequence tasks (summarization, translation) and extractive QA had their `pipeline` shortcuts removed in v5, so those use the `AutoModel` API directly — the same `from_pretrained` pattern, just a couple more lines. Each section's code shows you which.

## Setup
Find a partner. **Pick ONE task** from the list below. Run the code, then experiment with custom inputs. Report findings to the class.

## Time
15 minutes (depth, not breadth — go deep on ONE task).

## Tasks to choose from
- Named Entity Recognition (NER) — `pipeline`
- Question Answering — `AutoModel`
- Translation — `AutoModel`
- Summarization — `AutoModel`
- Fill-Mask — `pipeline`
- Text Generation — `pipeline`

Each task is in a section below. **Do ONE.** Don't try all of them — go deep, not broad.

In [ ]:
from transformers import pipeline

## Task 1: Named Entity Recognition (NER)

Identify entities in text — people, organizations, locations, etc.

**Run the code, then try your own text.** What does it get right? What does it miss?

In [ ]:
ner = pipeline('ner', aggregation_strategy='simple')

text = (
    'Sundar Pichai, CEO of Google based in Mountain View, met with the President '
    'in Washington last Tuesday to discuss AI policy.'
)

results = ner(text)
for r in results:
    print(f'{r["entity_group"]:6s} | {r["word"]:30s} (score: {r["score"]:.3f})')

# Try your own text
your_text = '...'
# results = ner(your_text)

## Task 2: Question Answering

Given a passage and a question, extract the answer.

In [ ]:
# transformers v5 removed the 'question-answering' pipeline shortcut.
# Extractive QA is an ENCODER-ONLY task: the model predicts the START and END token
# of the answer span inside the context. We run the model and decode that span.
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

qa_name = 'distilbert-base-cased-distilled-squad'
qa_tok = AutoTokenizer.from_pretrained(qa_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_name)

context = (
    'The Transformer architecture was introduced in 2017 by researchers at Google. '
    'It replaced recurrent neural networks for most NLP tasks. The key innovation '
    'was the attention mechanism, which allowed parallel processing of sequences. '
    'BERT, released by Google in 2018, is built on this architecture.'
)

questions = [
    'When was the Transformer introduced?',
    'Who introduced BERT?',
    'What did Transformers replace?',
]

def answer_question(question, context):
    inputs = qa_tok(question, context, return_tensors='pt', truncation=True, max_length=384)
    with torch.no_grad():
        outputs = qa_model(**inputs)
    start = outputs.start_logits.argmax()
    # Force the end token to come at/after the start so the span is never empty.
    end = outputs.end_logits[0, start:].argmax() + start + 1
    span = inputs['input_ids'][0][start:end]
    return qa_tok.decode(span, skip_special_tokens=True)

for q in questions:
    print(f'Q: {q}\nA: {answer_question(q, context)}\n')

## Task 3: Translation

English to other languages and back. Try multiple languages.

In [ ]:
# transformers v5 removed the 'translation_en_to_fr' pipeline shortcut.
# Translation is an ENCODER-DECODER (seq2seq) task. T5 is "text-to-text": the
# 'translate English to French:' prefix tells the SAME model to translate.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tr_name = 'google-t5/t5-small'
tr_tok = AutoTokenizer.from_pretrained(tr_name)
tr_model = AutoModelForSeq2SeqLM.from_pretrained(tr_name)

text = 'Machine learning is transforming many industries.'
inputs = tr_tok('translate English to French: ' + text, return_tensors='pt')
translated = tr_model.generate(**inputs, max_new_tokens=60, num_beams=4)
print(f'EN: {text}')
print(f'FR: {tr_tok.decode(translated[0], skip_special_tokens=True)}')

# Same model, just change the prefix — try other language pairs:
#   'translate English to German: ...'
#   'translate English to Romanian: ...'

## Task 4: Summarization

Condense long text into shorter summaries.

In [ ]:
# transformers v5 removed the 'summarization' pipeline shortcut.
# Summarization is an ENCODER-DECODER (seq2seq) task. T5 is "text-to-text":
# prefix the input with 'summarize: ' to tell the model what to do.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

sum_name = 'google-t5/t5-small'
sum_tok = AutoTokenizer.from_pretrained(sum_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(sum_name)

long_text = '''
Artificial intelligence has experienced unprecedented growth in capability over the past decade.
Beginning with the 2012 ImageNet breakthrough using deep convolutional neural networks, the field
has accelerated through several major milestones. The 2017 introduction of the Transformer
architecture enabled new possibilities in natural language processing. By 2018, BERT had set new
benchmarks across NLP tasks. GPT-3 in 2020 demonstrated emergent abilities at scale, performing
tasks it was never explicitly trained for. ChatGPT, launched in November 2022, brought conversational
AI to mainstream attention with over 100 million users in two months. Concurrently, diffusion models
transformed image generation, with Stable Diffusion making text-to-image freely available.
These advances have spurred massive industry investment, ethical debates, and regulatory responses.
'''

inputs = sum_tok('summarize: ' + long_text, return_tensors='pt', truncation=True, max_length=512)
summary_ids = sum_model.generate(**inputs, max_new_tokens=80, num_beams=4)
summary_text = sum_tok.decode(summary_ids[0], skip_special_tokens=True)
print(f'Original length: {len(long_text)} chars')
print(f'Summary length: {len(summary_text)} chars')
print(f'\nSummary:\n{summary_text}')

## Task 5: Fill-Mask (BERT's original training task)

Mask one word; the model predicts what fits.

In [ ]:
fill_mask = pipeline('fill-mask')

examples = [
    f'The cat sat on the {fill_mask.tokenizer.mask_token}.',
    f'Machine learning is changing the {fill_mask.tokenizer.mask_token}.',
    f'In 2017, the {fill_mask.tokenizer.mask_token} architecture was introduced.',
]

for text in examples:
    print(f'\nInput: {text}')
    results = fill_mask(text)
    for r in results[:5]:
        print(f'  {r["token_str"]:15s} (score: {r["score"]:.3f})')

## Task 6: Text Generation (sneak peek of Week 11)

Use a small decoder-only model to continue text. **This is what next week is about.**

In [ ]:
generator = pipeline('text-generation', model='gpt2')

prompts = [
    'The future of artificial intelligence is',
    'In 2030, machine learning will',
    'The most important skill for a programmer is',
]

for prompt in prompts:
    result = generator(prompt, max_new_tokens=40, do_sample=True, top_p=0.9)
    print(f'\nPrompt: {prompt}')
    print(f'Generated: {result[0]["generated_text"]}')

## Discussion (with partner — 5 minutes)

**Pick the task you tried in depth and discuss:**

1. **What did the model get right?** Specific examples.
2. **What did it get WRONG or do badly?** Specific examples. Why might it have failed?
3. **How would you describe this task in one sentence to a non-technical colleague?**
4. **What would you NOT use this for?** What are its limits?
5. **If you wanted to deploy this in production**, what's missing?

## Cross-task observations

All these tasks share the same architecture (transformer) and the same library (Hugging Face). Whether you call the `pipeline()` shortcut or load an `AutoModel` directly, it's the **same `from_pretrained` pattern** — different pretrained weights, one consistent API. **The unification is the point.**

## Bonus: Try YOUR pipeline

Look up a different task on Hugging Face Hub: https://huggingface.co/tasks. Pick something you haven't tried yet (text classification on a specific topic, image-to-text, audio classification, etc.). Get it working in under 10 lines of code. That's the goal.